In [1]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [4]:
from langchain_chroma import Chroma
# from langchain_openai import OpenAIEmbeddings
from langchain_ollama import OllamaEmbeddings

# inint embeddings model
embeddings = OllamaEmbeddings(model="llama3.1")
# embedding from documents object
vectorstore = Chroma.from_documents(
    documents,
    embedding=embeddings,
    # embedding=OllamaEmbeddings(model='llama3', base_url='http://125.69.16.175:11434'),
    
)

------------------------
http://127.0.0.1:11434/api/embed
------------------------


## Basic Initialization

In [5]:
vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

## Initialization from client

In [16]:
import chromadb

persistent_client = chromadb.PersistentClient()
collection = persistent_client.get_or_create_collection("collection_name")
collection.add(ids=["1", "2", "3"], documents=["a", "b", "c"])

vector_store_from_client = Chroma(
    client=persistent_client,
    collection_name="collection_name",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db"
)

Insert of existing embedding ID: 1
Insert of existing embedding ID: 2
Insert of existing embedding ID: 3
Add of existing embedding ID: 1
Add of existing embedding ID: 2
Add of existing embedding ID: 3


In [7]:
from uuid import uuid4

from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
    id=1,
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
    id=2,
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
    id=3,
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
    id=4,
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
    id=5,
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
    id=6,
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
    id=7,
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
    id=8,
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
    id=9,
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
    id=10,
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

------------------------
http://127.0.0.1:11434/api/embed
------------------------


['70f13c10-71bc-4c7f-8914-942085879c12',
 '11ac6e71-7941-450c-841a-3374ccdfa58c',
 'b81414ff-9740-4b77-96e2-4dc2d6181ce0',
 '7779faca-6bb0-421c-82a0-219e6ac61599',
 '2eda66dd-9e5c-445d-a0c9-46f592576a28',
 'c03b5b4f-7a7c-466c-882d-b48cf67e9af6',
 'bddee945-d5cc-404f-ab5f-a086af8d64a3',
 '7a1689a7-2d06-4f72-9554-1c279b151782',
 'cf570f04-d689-4185-bf31-75d7aacb771f',
 'd5dc0d6b-dab0-429a-aa81-69cc3bf34d65']

## Update items in vector store

In [8]:
updated_document_1 = Document(
    page_content="I had chocolate chip pancakes and fried eggs for breakfast this morning.",
    metadata={"source": "tweet"},
    id=1,
)

updated_document_2 = Document(
    page_content="The weather forecast for tomorrow is sunny and warm, with a high of 82 degrees.",
    metadata={"source": "news"},
    id=2,
)

vector_store.update_document(document_id=uuids[0], document=updated_document_1)
# You can also update multiple documents at once
vector_store.update_documents(
    ids=uuids[:2], documents=[updated_document_1, updated_document_2]
)

------------------------
http://127.0.0.1:11434/api/embed
------------------------
------------------------
http://127.0.0.1:11434/api/embed
------------------------


## Query vector store

In [12]:
vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

In [13]:
#  Query directly

results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": "tweet"},
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

------------------------
http://127.0.0.1:11434/api/embed
------------------------
* LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]
* Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]


In [14]:
results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow?", k=1, filter={"source": "news"}
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

------------------------
http://127.0.0.1:11434/api/embed
------------------------
* [SIM=0.621252] The weather forecast for tomorrow is sunny and warm, with a high of 82 degrees. [{'source': 'news'}]


In [5]:
vectorstore.similarity_search("cat")

------------------------
http://125.69.16.175:11434/api/embed
------------------------


[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.')]

In [8]:
embedding = OllamaEmbeddings(model="qwen2:7b", base_url='http://125.69.16.175:11434').embed_query("cat")

vectorstore.similarity_search_by_vector(embedding)

------------------------
http://125.69.16.175:11434/api/embed
------------------------


[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.')]

In [9]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)  # select top result

retriever.batch(["cat", "fish"])

------------------------
http://125.69.16.175:11434/api/embed
------------------------
------------------------
http://125.69.16.175:11434/api/embed
------------------------


[[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')],
 [Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]]

In [10]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

retriever.batch(["cat", "fish"])

------------------------
http://125.69.16.175:11434/api/embed
------------------------
------------------------
http://125.69.16.175:11434/api/embed
------------------------


[[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')],
 [Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]]

In [11]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="qwen2:7b", base_url='http://125.69.16.175:11434'
    # temperature=0,
    # other params...
)

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

{context}
"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | model

In [15]:
response = rag_chain.invoke("tell me about cats")

print(response.content)

------------------------
http://125.69.16.175:11434/api/embed
------------------------
------------------------
http://125.69.16.175:11434/api/chat
------------------------
The provided document focuses on dogs rather than cats. However, in general, cats are popular as pets due to their independent nature, adaptability, and affection when desired. They require less attention and care compared to some other animals, making them ideal for busy lifestyles or for those who prefer more solitary companionship. Cats can be very clean and often groom themselves, which reduces the need for frequent baths or grooming sessions compared to dogs. They also have unique vocalizations that vary among different breeds and individual cats, ranging from meows to purrs. Some cat owners enjoy training their cats for tricks or engaging in interactive play, while others prefer a more laid-back relationship with their feline friends. Cats come in many sizes and colors, providing options for various preference